# Generate Test Sentences with TTS Models

In [ ]:
!pip install -U -q soundfile pandas
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision torchcodec
!pip install --no-cache-dir --no-dependencies "kittenml==0.9.8"
!pip install misaki num2words spacy onnxruntime espeakng-loader

## Load the Problem Words

In [2]:
import pandas as pd

problem_words = pd.read_csv("problem_words.csv", sep="\t")
problem_words.head(20)

,word,error_count,example_label,example_prediction,example_source
0,cristiano ronaldo,1,cristiano ronaldo has won twenty four trophies...,it is carried and the brand has been wanted tw...,granite_finetuned_commonvoice_without_context
1,twenty four,1,cristiano ronaldo has won twenty four trophies...,it is carried and the brand has been wanted tw...,granite_finetuned_commonvoice_without_context
2,john wesley hardin,1,he was later killed by john wesley hardin duri...,he was later killed by john wesley harding dur...,granite_finetuned_commonvoice_without_context
3,suttontaylor,1,he was later killed by john wesley hardin duri...,he was later killed by john wesley harding dur...,granite_finetuned_commonvoice_without_context
4,steve lane,1,she married her business partner steve lane,she married her business partner steven lee,granite_finetuned_commonvoice_without_context
5,groves,1,groves started writing songs when she was four...,grafs started writing songs when she was four ...,granite_finetuned_commonvoice_without_context
6,kris taylor,1,manager dave ross brought in kris taylor to fo...,manager dave ross brought in chris taylor to f...,granite_finetuned_commonvoice_without_context
7,missus hazelton,1,there could not be a better time for alexis to...,there could not be a better time for alexis to...,granite_finetuned_librispeech_without_context
8,sybal,1,he would rather read the letter in the street ...,he would rather read the letter in the street ...,granite_finetuned_librispeech_without_context
9,nieaner,1,and a wellspring of heartburning and annoyance...,and a wellspring of heartburning and annoyance...,granite_finetuned_librispeech_without_context


## Build Test Sentences

In [3]:
#TODO: ADD YOUR OWN IMPLEMENTATION FOR EXAMPLE SENTENCES
def build_sentence(word):
    return f"Here is the word {word}. The word {word} is often mispronounced."

TOP_N = 1000   # synthesize all problem words, capped at 1000
sentences = [(w, build_sentence(w)) for w in problem_words["word"].head(TOP_N)]

for w, s in sentences[:5]:
    print(f"{w!r}: {s}")

'cristiano ronaldo': Here is the word cristiano ronaldo. The word cristiano ronaldo is often mispronounced.
'twenty four': Here is the word twenty four. The word twenty four is often mispronounced.
'john wesley hardin': Here is the word john wesley hardin. The word john wesley hardin is often mispronounced.
'suttontaylor': Here is the word suttontaylor. The word suttontaylor is often mispronounced.
'steve lane': Here is the word steve lane. The word steve lane is often mispronounced.


## TTS Model 1 : KittenTTS

In [4]:
from kittenml import KittenTTS

kitten = KittenTTS("KittenML/kitten-tts-2-mini")
print(kitten.available_voices)

for v in kitten.voices("en"):
    print(v["name"], "-", v["description"])

{'en': ['Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent', 'Max', 'Danny'], 'de': ['Lena', 'Greta', 'Jonas', 'Felix', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'es': ['Lucia', 'Carmen', 'Javier', 'Diego', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'it': ['Giulia', 'Chiara', 'Marco', 'Luca', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'pt': ['Beatriz', 'Camila', 'Rafael', 'Thiago', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'ru': ['Anastasia', 'Natasha', 'Dmitri', 'Ivan', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'hi': ['Priya', 'Ananya', 'Arjun', 'Vikram'], 'da': ['Freja', 'Mads', 'Rasmus'], 'fi': ['Aino', 'Helmi', 'Eero', 'Juho', 'Ville'], 'fr': ['Camille', 'Chloe', 'Manon', 'Louis', 'Antoine', 'Julien', 'Mathis', 'Etienne'], 'ja': ['Yuki'

In [5]:
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

OUTPUT_DIR = Path("generated")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KITTEN_VOICE = "Eleanor"   # <-- choose a voice from the list above

for w, sentence in sentences:
    audio = kitten.generate(sentence, voice=KITTEN_VOICE)
    out_path = OUTPUT_DIR / f"{w}.wav"
    sf.write(out_path.as_posix(), audio, 24000)

# listen to the first generated sentence
display(Audio(str(OUTPUT_DIR / f"{sentences[0][0]}.wav")))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

/opt/conda/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Build the Synthesized Dataset (Manifest + Train/Dev Split)

In [6]:
import random

# one manifest row per synthesized sentence: audio path, the entity word, the transcript
dataset = []
for w, sentence in sentences:
    wav_path = OUTPUT_DIR / f"{w}.wav"
    if wav_path.exists():
        dataset.append({"audio": wav_path.as_posix(), "word": w, "text": sentence})

random.seed(42)
random.shuffle(dataset)

DEV_FRACTION = 0.2   # fraction of the synthesized data held out for evaluation
n_dev = max(1, int(len(dataset) * DEV_FRACTION))
dev_split, train_split = dataset[:n_dev], dataset[n_dev:]

for name, split in [("synthesized_train.tsv", train_split), ("synthesized_dev.tsv", dev_split)]:
    with open(name, "w", encoding="utf-8") as f:
        for row in split:
            f.write(f"{row['audio']}\t{row['word']}\t{row['text']}\n")

print(f"{len(train_split)} train / {len(dev_split)} dev sentences written to synthesized_train.tsv / synthesized_dev.tsv")

11 train / 2 dev sentences written to synthesized_train.tsv / synthesized_dev.tsv


## Listening Task : Are the Problem Words Spoken Correctly?

In [7]:
# Listen to the generated sentences for the top problem words.
# For each word, judge: is the word spoken correctly and clearly?

for w, sentence in sentences[:5]:
    print("=" * 70)
    print("WORD     :", w)
    print("SENTENCE :", sentence)

    display(Audio(str(OUTPUT_DIR / f"{w}.wav")))


WORD     : cristiano ronaldo
SENTENCE : Here is the word cristiano ronaldo. The word cristiano ronaldo is often mispronounced.


WORD     : twenty four
SENTENCE : Here is the word twenty four. The word twenty four is often mispronounced.


WORD     : john wesley hardin
SENTENCE : Here is the word john wesley hardin. The word john wesley hardin is often mispronounced.


WORD     : suttontaylor
SENTENCE : Here is the word suttontaylor. The word suttontaylor is often mispronounced.


WORD     : steve lane
SENTENCE : Here is the word steve lane. The word steve lane is often mispronounced.


WORD     : groves
SENTENCE : Here is the word groves. The word groves is often mispronounced.


... (3 outputs left out)

WORD     : nieaner
SENTENCE : Here is the word nieaner. The word nieaner is often mispronounced.


WORD     : sidney
SENTENCE : Here is the word sidney. The word sidney is often mispronounced.


WORD     : jane dimond
SENTENCE : Here is the word jane dimond. The word jane dimond is often mispronounced.


WORD     : one
SENTENCE : Here is the word one. The word one is often mispronounced.
